# Expand AATC

In https://drivenbydata.atlassian.net/browse/DD-2117 it is requested that the AATC is expanded, so that AAT concepts submitted by PAN, do not fail, if the submitted AAT concepts are valid concepts (gpv:Concept and not gpv:GuideTerm)


## How large will AATC get?

If, we include:
* concepts with EN, and no NL labels
* obsolete concepts

## development steps

* create test data - because the SPARQL query in [aatc_generate.py](aatc_generate.py) is a bit finicky, and the amount of return data is large, I would like to start by assembling test data
    * containing: 1 concept with only @en label, 1 concept with @nl and @en labels and one concept with @en-US label 
* 1) write select query: select all concepts with @en or @en-US labels and include also @nl labels when existing
* write construct query
* test construct results
* run construct query against getty SPARQL end-point
* count number of resulting triples
* test results


In [14]:
#### boiler plate functions ####
# imports SPARQL prefixes and functions defs

import os
from pprint import pprint
from SPARQLWrapper import SPARQLWrapper, JSON, TURTLE, CSV 

prefixes = '''    
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX aat: <http://vocab.getty.edu/aat/>
PREFIX gvp: <http://vocab.getty.edu/ontology#> 
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX skosxl: <http://www.w3.org/2008/05/skos-xl#>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX aatc: <http://vocabularies.dans.knaw.nl/aatconcepts/>
'''    


def sparql_endpoint_query(query, format, endpoint="http://vocab.getty.edu/sparql"):
    formats = {"json": JSON, "turtle": TURTLE, "csv": CSV}
    f_ = formats[format]
    endpoint = endpoint 
    sparql = SPARQLWrapper(endpoint)

    query = prefixes + query     
    sparql.setQuery(query)

    sparql.setReturnFormat(f_)
    results = sparql.query().convert()    
    # sparql.setReturnFormat(XML)
    # results = sparql.query()
    return results
    # # Print the results
    # print("Subject ID: ", subjectID)
    # 


def print_sparql_results(results):
    for row in results["results"]["bindings"]:
        return (row)


In [15]:
#### CONSTRUCT query to create test data test/sample-data.ttl ####
# including the concept & all its skosxl:prefLabel nodes, 
# as well as the skosxl:prefLabel nodes' property value pairs

# @en http://vocab.getty.edu/aat/300189559 and @nl
# @en-US http://vocab.getty.edu/aat/300022441 and @nl (no @en label)
# @en only no Dutch http://vocab.getty.edu/aat/300456698'

query_construct = '''
CONSTRUCT
{     ?concept a gvp:Concept ;
               skos:inScheme aat: .   

      ?concept skosxl:prefLabel ?preflabel .
      ?preflabel ?preflabelProp ?preflabelVal .
} 
WHERE 
{
   VALUES ?concept { aat:300189559 aat:300022441 aat:300456698 } 

   ?concept a gvp:Concept .
   ?concept skosxl:prefLabel ?preflabel.
   ?preflabel ?preflabelProp ?preflabelVal .
}
'''
testq = sparql_endpoint_query(query=query_construct, format='turtle')
with open('tests/sample-data.ttl', 'wb') as sampledata:
      sampledata.write(testq)


In [ ]:
#### QUERY Test Data #### 

In the next section of code, I will count how many concepts the aatc.ttl contains.
I will be running [aatc_generate.py](aatc_generate.py) beforehand to get some updates, which might make aatc larger   

In [58]:
# count how many concepts the aatc.ttl contains

from rdflib import Graph

# Load your RDF data
g = Graph()
g.parse("aatc.ttl", format="ttl")  # can also use "ttl" for Turtle

# SPARQL query to count all foaf:Person
query = """
  SELECT (COUNT(?concept) AS ?conceptCount) 
    WHERE {
      ?concept a skos:Concept .
    }
"""
query = prefixes + query
results = g.query(query)


In [57]:

results.serialize()
# 55915 concepts in AATC.ttl


b'<?xml version="1.0" encoding="utf-8"?>\n<sparql xmlns="http://www.w3.org/2005/sparql-results#" xmlns:xml="http://www.w3.org/XML/1998/namespace"><head><variable name="conceptCount"></variable></head><results><result><binding name="conceptCount"><literal datatype="http://www.w3.org/2001/XMLSchema#integer">55915</literal></binding></result></results></sparql>'

The following code block will run the same AAT query, as used in  [aatc_generate.py](aatc_generate.py), but **counting** the number of concepts with labels in EN and NL.

The number (98439) is larger than that of aatc, as this concepts, can appear as individuals

In [56]:
# SELECT QUERY: GETTY AAT Concepts with labels in EN and NL - same query as aatc_generate.py
# lang: @en-US is converted to @en 

pref_label_query = '''
SELECT (COUNT(?concept) AS ?conceptCount) 
WHERE {
    ?concept a gvp:Concept ;
        skos:inScheme aat: ;
        skosxl:prefLabel ?preflabel .
        
    OPTIONAL {?concept dcterms:issued ?issuedDate .}

    { ?preflabel  dcterms:language aat:300388277 ; skosxl:literalForm ?label_literal_en.  } # lang: @en
                 
    UNION

    { ?preflabel  dcterms:language aat:300387822 ; skosxl:literalForm ?label_literal_en_us . 
      BIND (STRLANG(STR(?label_literal_en_us), 'en') AS  ?label_literal_en)} # lang: @en-US - converts to @en
    
    UNION
    
    { ?preflabel  dcterms:language aat:300388256 ; skosxl:literalForm ?label_literal_nl . } # lang: @nl 
    FILTER( STRSTARTS(str(?concept), str(aat:)) )
}
''' 
total_concepts_EN_NL = sparql_endpoint_query(query=pref_label_query, format='json')
print(f"Total of AAT Concepts with NL and EN labels: {total_concepts_EN_NL['results']['bindings'][0]['conceptCount']['value']}")

Total of AAT Concepts with NL and EN labels: 98439


See [lang_queries.rq](lang_queries.rq)

In [59]:
# SELECT QUERY: GETTY AAT Concepts with labels in EN and NL - same query as aatc_generate.py
# lang: @en-US is converted to @en 

pref_label_query = '''
SELECT (COUNT(?concept) AS ?conceptCount) 
WHERE {
    ?concept a gvp:Concept ;
        skos:inScheme aat: ;
        skosxl:prefLabel ?preflabel .
        
    OPTIONAL {?concept dcterms:issued ?issuedDate .}

    { ?preflabel  dcterms:language aat:300388277 ; skosxl:literalForm ?label_literal_en.  } # lang: @en
                 
    UNION

    { ?preflabel  dcterms:language aat:300387822 ; skosxl:literalForm ?label_literal_en_us . 
      BIND (STRLANG(STR(?label_literal_en_us), 'en') AS  ?label_literal_en)} # lang: @en-US - converts to @en
    
    UNION
    
    { ?preflabel  dcterms:language aat:300388256 ; skosxl:literalForm ?label_literal_nl . } # lang: @nl 

    FILTER( STRSTARTS(str(?concept), str(aat:)) )
}
''' 
total_concepts_EN_NL = sparql_endpoint_query(query=pref_label_query, format='json')
print(f"Total of AAT Concepts with NL and EN labels: {total_concepts_EN_NL['results']['bindings'][0]['conceptCount']['value']}")

EndPointInternalError: EndPointInternalError: The endpoint returned the HTTP status code 500.